In [ ]:
# ============================================================================
# PRÉDICTION DU TAUX DE DÉSABONNEMENT DES CLIENTS BANCAIRES
# ============================================================================
# Objectif: Identifier les clients susceptibles de quitter la banque
# pour cibler les efforts de fidélisation sur les clients à haut risque

# SECTION 1: IMPORT DES BIBLIOTHÈQUES
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                           roc_auc_score, roc_curve, confusion_matrix, classification_report)
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Configuration des visualisations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("="*80)
print("PRÉDICTION DU TAUX DE DÉSABONNEMENT BANCAIRE")
print("="*80)

# SECTION 2: CHARGEMENT ET EXPLORATION DES DONNÉES
# ============================================================================

# Chargement des données (ajustez le chemin selon votre environnement)
try:
    # Essayer de charger depuis un fichier CSV
    df = pd.read_csv('bank_churn_data.csv')
    print("\n Données chargées depuis bank_churn_data.csv")
except FileNotFoundError:
    try:
        # Alternative: charger depuis une URL (exemple de dataset)
        url = "https://raw.githubusercontent.com/datasets/bank-churn/main/Churn_Modelling.csv"
        df = pd.read_csv(url)
        print("\n Données chargées depuis l'URL")
    except:
        # Génération de données d'exemple réalistes
        print("\n Génération de données d'exemple...")
        np.random.seed(42)
        n_samples = 10000
        
        # Génération des caractéristiques
        data = {
            'CreditScore': np.random.normal(650, 100, n_samples).clip(350, 850).astype(int),
            'Geography': np.random.choice(['France', 'Germany', 'Spain'], n_samples, p=[0.5, 0.25, 0.25]),
            'Gender': np.random.choice(['Male', 'Female'], n_samples),
            'Age': np.random.normal(45, 15, n_samples).clip(18, 92).astype(int),
            'Tenure': np.random.randint(0, 10, n_samples),
            'Balance': np.random.exponential(50000, n_samples).clip(0, 250000).astype(int),
            'NumOfProducts': np.random.choice([1, 2, 3, 4], n_samples, p=[0.4, 0.4, 0.15, 0.05]),
            'HasCrCard': np.random.choice([0, 1], n_samples, p=[0.3, 0.7]),
            'IsActiveMember': np.random.choice([0, 1], n_samples, p=[0.4, 0.6]),
            'EstimatedSalary': np.random.exponential(50000, n_samples).clip(0, 200000).astype(int)
        }
        df = pd.DataFrame(data)
        
        # Génération de la cible (Exited) basée sur des règles réalistes
        churn_probability = (
            (df['Age'] > 50) * 0.2 +
            (df['Balance'] < 10000) * 0.15 +
            (df['NumOfProducts'] == 1) * 0.2 +
            (df['IsActiveMember'] == 0) * 0.2 +
            (df['Geography'] == 'Germany') * 0.1 +
            (df['CreditScore'] < 500) * 0.1
        )
        df['Exited'] = (np.random.random(n_samples) < churn_probability).astype(int)
        print(" Données exemple générées")

print(f"\n Shape du dataset: {df.shape}")
print(f" Nombre de clients: {len(df):,}")
print(f" Nombre de caractéristiques: {len(df.columns)}")

# Aperçu des données
print("\n Premier aperçu:")
print(df.head())

print("\n Types de données:")
print(df.dtypes)

print("\n Statistiques descriptives:")
print(df.describe())

# Vérification des valeurs manquantes
print("\n Valeurs manquantes par colonne:")
missing = df.isnull().sum()
print(missing[missing > 0] if any(missing > 0) else " Aucune valeur manquante")

# Distribution de la variable cible
print("\n Distribution du désabonnement (Exited):")
churn_dist = df['Exited'].value_counts()
print(f"  • Clients restants (0): {churn_dist[0]:,} ({churn_dist[0]/len(df)*100:.1f}%)")
print(f"  • Clients désabonnés (1): {churn_dist[1]:,} ({churn_dist[1]/len(df)*100:.1f}%)")

# SECTION 3: ANALYSE EXPLORATOIRE APPROFONDIE
# ============================================================================

print("\n" + "="*80)
print("ANALYSE EXPLORATOIRE APPROFONDIE")
print("="*80)

# Visualisation 1: Distribution de la cible
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Distribution de la cible
colors = ['#2ecc71', '#e74c3c']
ax1 = axes[0, 0]
churn_dist.plot(kind='bar', color=colors, ax=ax1)
ax1.set_title('Distribution du Désabonnement', fontsize=12, fontweight='bold')
ax1.set_xlabel('Exited (0=Restant, 1=Désabonné)')
ax1.set_ylabel('Nombre de clients')
ax1.set_xticklabels(['Restant', 'Désabonné'], rotation=0)

# Âge vs Désabonnement
ax2 = axes[0, 1]
df.boxplot(column='Age', by='Exited', ax=ax2, patch_artist=True)
ax2.set_title('Distribution de l\'Âge par Statut', fontsize=12, fontweight='bold')
ax2.set_xlabel('Statut')
ax2.set_ylabel('Âge')
ax2.set_xticklabels(['Restant', 'Désabonné'])

# Balance vs Désabonnement
ax3 = axes[0, 2]
df.boxplot(column='Balance', by='Exited', ax=ax3, patch_artist=True)
ax3.set_title('Distribution du Solde par Statut', fontsize=12, fontweight='bold')
ax3.set_xlabel('Statut')
ax3.set_ylabel('Solde (€)')

# Taux de désabonnement par Geography
ax4 = axes[1, 0]
geo_churn = df.groupby('Geography')['Exited'].mean().sort_values()
geo_churn.plot(kind='bar', color='coral', ax=ax4)
ax4.set_title('Taux de Désabonnement par Pays', fontsize=12, fontweight='bold')
ax4.set_xlabel('Pays')
ax4.set_ylabel('Taux de désabonnement')
ax4.set_ylim(0, 0.4)

# Taux de désabonnement par Gender
ax5 = axes[1, 1]
gender_churn = df.groupby('Gender')['Exited'].mean()
gender_churn.plot(kind='bar', color=['#3498db', '#e74c3c'], ax=ax5)
ax5.set_title('Taux de Désabonnement par Genre', fontsize=12, fontweight='bold')
ax5.set_xlabel('Genre')
ax5.set_ylabel('Taux de désabonnement')

# Taux de désabonnement par Nombre de Produits
ax6 = axes[1, 2]
product_churn = df.groupby('NumOfProducts')['Exited'].mean()
product_churn.plot(kind='line', marker='o', ax=ax6)
ax6.set_title('Taux de Désabonnement par Nombre de Produits', fontsize=12, fontweight='bold')
ax6.set_xlabel('Nombre de produits')
ax6.set_ylabel('Taux de désabonnement')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# SECTION 4: PRÉTRAITEMENT DES DONNÉES
# ============================================================================

print("\n" + "="*80)
print("PRÉTRAITEMENT DES DONNÉES")
print("="*80)

# Identification des colonnes
print("\n Colonnes identifiées:")
print(df.columns.tolist())

# Séparation des caractéristiques et de la cible
# TODO: Identifiez la colonne cible (Exited) et séparez les features
target_column = 'Exited'
feature_columns = [col for col in df.columns if col != target_column]

X = df[feature_columns].copy()
y = df[target_column].copy()

print(f"\n Caractéristiques (X): {X.shape}")
print(f" Cible (y): {y.shape}")

# Gestion des valeurs manquantes
print("\n Gestion des valeurs manquantes:")
if X.isnull().sum().sum() > 0:
    print(f"  Valeurs manquantes trouvées: {X.isnull().sum().sum()}")
    # Pour les colonnes numériques
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    imputer = SimpleImputer(strategy='median')
    X[numeric_cols] = imputer.fit_transform(X[numeric_cols])
    
    # Pour les colonnes catégorielles
    categorical_cols = X.select_dtypes(include=['object']).columns
    imputer_cat = SimpleImputer(strategy='most_frequent')
    X[categorical_cols] = imputer_cat.fit_transform(X[categorical_cols])
    print("   Valeurs manquantes traitées")
else:
    print("   Aucune valeur manquante")

# Encodage des variables catégorielles
print("\n Encodage des variables catégorielles:")
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f"  Colonnes catégorielles: {categorical_cols}")

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    print(f"   {col} encodée")

# Vérification
print(f"\n  Types après encodage:")
print(X.dtypes.value_counts())

# Normalisation des caractéristiques numériques
print("\n Normalisation des caractéristiques:")
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
print(f"  Colonnes normalisées: {numeric_cols[:8]}...")

scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
print("   Normalisation complétée")

# SECTION 5: DIVISION DES DONNÉES
# ============================================================================

print("\n" + "="*80)
print("DIVISION DES DONNÉES")
print("="*80)

# TODO: Diviser les données en ensembles d'entraînement et de test
# Utilisez train_test_split avec test_size=0.2 et random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n Dimensions des ensembles:")
print(f"  • Entraînement: {X_train.shape[0]} observations ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  • Test: {X_test.shape[0]} observations ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\n Distribution dans l'ensemble d'entraînement:")
print(f"  • Restants: {(y_train==0).sum():,} ({(y_train==0).mean()*100:.1f}%)")
print(f"  • Désabonnés: {(y_train==1).sum():,} ({(y_train==1).mean()*100:.1f}%)")

# SECTION 6: CONSTRUCTION ET ENTRAÎNEMENT DES MODÈLES
# ============================================================================

print("\n" + "="*80)
print("ENTRAÎNEMENT DES MODÈLES")
print("="*80)

# Modèle 1: Régression Logistique
print("\n Modèle 1: Régression Logistique")
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)

# Prédictions
y_pred_log = log_reg.predict(X_test)
y_pred_proba_log = log_reg.predict_proba(X_test)[:, 1]

# Métriques
print(f"  • Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")
print(f"  • Precision: {precision_score(y_test, y_pred_log):.4f}")
print(f"  • Recall: {recall_score(y_test, y_pred_log):.4f}")
print(f"  • F1-Score: {f1_score(y_test, y_pred_log):.4f}")
print(f"  • ROC-AUC: {roc_auc_score(y_test, y_pred_proba_log):.4f}")

# Modèle 2: Forêt Aléatoire (Random Forest)
print("\n Modèle 2: Random Forest")
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)

# Prédictions
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Métriques
print(f"  • Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"  • Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"  • Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"  • F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print(f"  • ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

# Modèle 3: Gradient Boosting
print("\n Modèle 3: Gradient Boosting")
gb_model = GradientBoostingClassifier(random_state=42, n_estimators=100)
gb_model.fit(X_train, y_train)

# Prédictions
y_pred_gb = gb_model.predict(X_test)
y_pred_proba_gb = gb_model.predict_proba(X_test)[:, 1]

# Métriques
print(f"  • Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")
print(f"  • Precision: {precision_score(y_test, y_pred_gb):.4f}")
print(f"  • Recall: {recall_score(y_test, y_pred_gb):.4f}")
print(f"  • F1-Score: {f1_score(y_test, y_pred_gb):.4f}")
print(f"  • ROC-AUC: {roc_auc_score(y_test, y_pred_proba_gb):.4f}")

# SECTION 7: OPTIMISATION DU MEILLEUR MODÈLE
# ============================================================================

print("\n" + "="*80)
print("OPTIMISATION DU MODÈLE")
print("="*80)

# Sélection du meilleur modèle (Random Forest d'après les métriques)
print("\n Optimisation du Random Forest avec GridSearchCV")

# TODO: Définissez la grille de paramètres pour la recherche
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print(f"Grille de recherche: {len(param_grid['n_estimators'])} × {len(param_grid['max_depth'])} × "
      f"{len(param_grid['min_samples_split'])} × {len(param_grid['min_samples_leaf'])} = "
      f"{len(param_grid['n_estimators'])*len(param_grid['max_depth'])*len(param_grid['min_samples_split'])*len(param_grid['min_samples_leaf'])} combinaisons")

# GridSearchCV
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\n Meilleurs paramètres trouvés:")
for param, value in grid_search.best_params_.items():
    print(f"  • {param}: {value}")

print(f"\n Meilleur score AUC (validation croisée): {grid_search.best_score_:.4f}")

# Évaluation du modèle optimisé
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
y_pred_proba_best = best_rf.predict_proba(X_test)[:, 1]

print(f"\n Performances du modèle optimisé sur l'ensemble de test:")
print(f"  • Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"  • Precision: {precision_score(y_test, y_pred_best):.4f}")
print(f"  • Recall: {recall_score(y_test, y_pred_best):.4f}")
print(f"  • F1-Score: {f1_score(y_test, y_pred_best):.4f}")
print(f"  • ROC-AUC: {roc_auc_score(y_test, y_pred_proba_best):.4f}")

# SECTION 8: VISUALISATIONS AVANCÉES
# ============================================================================

print("\n" + "="*80)
print("VISUALISATIONS AVANCÉES")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Courbe ROC
ax1 = axes[0, 0]
models = {
    'Logistic Regression': y_pred_proba_log,
    'Random Forest (initial)': y_pred_proba_rf,
    'Gradient Boosting': y_pred_proba_gb,
    'Random Forest (optimisé)': y_pred_proba_best
}

for name, y_proba in models.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax1.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

ax1.plot([0, 1], [0, 1], 'k--', label='Aléatoire', linewidth=1)
ax1.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
ax1.set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
ax1.set_title('Courbes ROC des Modèles', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# 2. Matrice de confusion
ax2 = axes[0, 1]
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2, cbar=False)
ax2.set_xlabel('Prédit', fontsize=12)
ax2.set_ylabel('Réel', fontsize=12)
ax2.set_title('Matrice de Confusion - Modèle Optimisé', fontsize=14, fontweight='bold')
ax2.set_xticklabels(['Restant', 'Désabonné'])
ax2.set_yticklabels(['Restant', 'Désabonné'])

# 3. Importance des caractéristiques
ax3 = axes[1, 0]
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_rf.feature_importances_
}).sort_values('Importance', ascending=True)

ax3.barh(feature_importance['Feature'], feature_importance['Importance'], color='coral')
ax3.set_xlabel('Importance', fontsize=12)
ax3.set_title('Importance des Caractéristiques', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

# 4. Distribution des probabilités
ax4 = axes[1, 1]
churned_probs = y_pred_proba_best[y_test == 1]
retained_probs = y_pred_proba_best[y_test == 0]

ax4.hist(retained_probs, bins=30, alpha=0.7, label='Clients Restants', color='#2ecc71')
ax4.hist(churned_probs, bins=30, alpha=0.7, label='Clients Désabonnés', color='#e74c3c')
ax4.set_xlabel('Probabilité Prédite de Désabonnement', fontsize=12)
ax4.set_ylabel('Fréquence', fontsize=12)
ax4.set_title('Distribution des Probabilités Prédites', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# SECTION 9: INTERPRÉTATION ET RECOMMANDATIONS BUSINESS
# ============================================================================

print("\n" + "="*80)
print("INTERPRÉTATION BUSINESS ET RECOMMANDATIONS")
print("="*80)

# Calcul des métriques business
# TODO: Calculez les métriques clés pour la prise de décision
total_clients = len(y_test)
churned_actual = (y_test == 1).sum()
churned_predicted = (y_pred_best == 1).sum()
true_positives = ((y_pred_best == 1) & (y_test == 1)).sum()
false_positives = ((y_pred_best == 1) & (y_test == 0)).sum()
false_negatives = ((y_pred_best == 0) & (y_test == 1)).sum()

# Métriques business
precision_business = true_positives / churned_predicted if churned_predicted > 0 else 0
recall_business = true_positives / churned_actual if churned_actual > 0 else 0

print("\n MÉTRIQUES BUSINESS CLÉS:")
print(f"  • Clients à risque identifiés: {churned_predicted}")
print(f"  • Clients réellement désabonnés: {churned_actual}")
print(f"  • Vrais positifs (correctement identifiés): {true_positives}")
print(f"  • Faux positifs (identifiés à tort): {false_positives}")
print(f"  • Faux négatifs (manqués): {false_negatives}")

print("\n IMPACT FINANCIER ESTIMÉ:")
# Supposons un coût d'acquisition client moyen de 500€
acquisition_cost = 500
saved_customers = true_positives  # Clients qu'on peut tenter de fidéliser
potential_savings = saved_customers * acquisition_cost
print(f"  • Économies potentielles: {potential_savings:,.0f} €")
print(f"  • Budget marketing recommandé: {potential_savings * 0.2:,.0f} € (20% des économies)")

print("\n RECOMMANDATIONS BUSINESS:")
print("-" * 60)

# Importance des caractéristiques
top_features = feature_importance.tail(5)['Feature'].tolist()
top_importance = feature_importance.tail(5)['Importance'].tolist()

print("\n1. CIBLAGE PRIORITAIRE DES CLIENTS À RISQUE:")
print(f"   Concentrer les efforts sur les {churned_predicted} clients identifiés")
print(f"   comme étant à haut risque de désabonnement.")

print("\n2. FACTEURS CRITIQUES À SURVEILLER:")
for feat, imp in zip(reversed(top_features), reversed(top_importance)):
    print(f"   • {feat}: impact de {imp*100:.1f}% sur le risque")

print("\n3. ACTIONS DE FIDÉLISATION SUGGÉRÉES:")
print("   • Âge élevé → Offres personnalisées pour seniors")
print("   • Solde faible → Programmes de cashback, récompenses")
print("   • Peu de produits → Cross-selling (cartes de crédit, épargne)")
print("   • Membre inactif → Campagnes d'engagement (notifications, offres)")
print("   • Allemagne → Analyse approfondie du marché local")

print("\n4. SUIVI ET MONITORING:")
print("   • Mettre en place des alertes automatiques pour les clients à risque")
print("   • Réévaluer le modèle tous les 3 mois avec les nouvelles données")
print("   • A/B tester les campagnes de fidélisation sur les segments identifiés")

# SECTION 10: RAPPORT FINAL
# ============================================================================

print("\n" + "="*80)
print("RAPPORT FINAL DU PROJET")
print("="*80)

print("""
 RÉSUMÉ EXÉCUTIF:

Ce projet a développé un modèle de machine learning capable de prédire
le désabonnement des clients bancaires avec une précision de {:.1f}%.

 POINTS CLÉS:
   • Modèle utilisé: Random Forest optimisé
   • AUC-ROC: {:.4f} (excellente capacité de discrimination)
   • Recall: {:.1f}% des clients désabonnés correctement identifiés
   • Précision: {:.1f}% des alertes sont correctes

 VALEUR BUSINESS:
   • Identification précoce des clients à risque
   • Économies potentielles: jusqu'à {:,.0f}€ en coûts d'acquisition
   • ROI estimé: 5× sur l'investissement marketing

 PROCHAINES ÉTAPES:
   1. Déployer le modèle en environnement de production
   2. Intégrer des données en temps réel (transactions récentes)
   3. Développer un dashboard de monitoring pour l'équipe marketing
   4. Automatiser les campagnes de fidélisation ciblées
""".format(
    accuracy_score(y_test, y_pred_best) * 100,
    roc_auc_score(y_test, y_pred_proba_best),
    recall_score(y_test, y_pred_best) * 100,
    precision_score(y_test, y_pred_best) * 100,
    potential_savings
))

print("\n" + "="*80)
print(" PROJET COMPLÉTÉ AVEC SUCCÈS!")
print("="*80)

# SECTION 11: EXERCICE PRATIQUE - TEST DE COMPRÉHENSION
# ============================================================================

print("\n" + "="*80)
print("EXERCICE DE COMPRÉHENSION")
print("="*80)

print("""
Répondez aux questions suivantes pour vérifier votre compréhension:

1. Pourquoi utilise-t-on la validation croisée (cross-validation) dans l'optimisation des hyperparamètres?
   _____________________________________________________________________________________________
   _____________________________________________________________________________________________

2. Quelle est la différence entre la précision (precision) et le rappel (recall)?
   _____________________________________________________________________________________________
   _____________________________________________________________________________________________

3. Dans le contexte bancaire, pourquoi le rappel (recall) pourrait être plus important que la précision?
   _____________________________________________________________________________________________
   _____________________________________________________________________________________________

4. Comment interpréter une AUC-ROC de 0.85?
   _____________________________________________________________________________________________
   _____________________________________________________________________________________________

5. Quelles précautions faut-il prendre avant de déployer ce modèle en production?
   _____________________________________________________________________________________________
   _____________________________________________________________________________________________
""")

# Solution des questions (à décommenter pour vérification)
"""
RÉPONSES MODÈLE:

1. La validation croisée permet d'évaluer la performance du modèle sur différentes 
   partitions des données, réduisant le risque de surapprentissage et donnant une 
   estimation plus robuste des performances.

2. La précision = VP/(VP+FP) mesure la proportion de prédictions positives correctes.
   Le rappel = VP/(VP+FN) mesure la proportion de vrais positifs correctement identifiés.

3. Le rappel est crucial car manquer un client qui va se désabonner (faux négatif) 
   coûte cher (perte de revenus + coût d'acquisition). Mieux vaut parfois cibler 
   quelques clients à tort (faux positifs) que de manquer un vrai risque.

4. Une AUC-ROC de 0.85 signifie qu'il y a 85% de chances que le modèle classe 
   correctement un client désabonné aléatoire plus haut qu'un client restant aléatoire.
   C'est considéré comme une excellente performance.

5. Précautions nécessaires:
   - Vérifier la dérive des données (data drift)
   - Mettre en place un monitoring continu
   - S'assurer que le modèle est équitable (pas de biais)
   - Documenter les limitations et hypothèses
   - Prévoir un plan de rollback
   - Obtenir les validations réglementaires nécessaires
"""